# 03 - Preprocessing
Penanganan missing values, outlier, normalisasi, dan pembagian data.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs('../models', exist_ok=True)

df = pd.read_csv('../data/processed/dataset_indonesia.csv')
print('Shape awal:', df.shape)
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Penanganan missing values dengan interpolasi linear
df_clean = df.copy()
df_clean = df_clean.set_index('Year')
df_clean = df_clean.interpolate(method='linear', limit_direction='both')
df_clean = df_clean.reset_index()

print('Missing setelah interpolasi:')
print(df_clean.isnull().sum())

# Drop baris yang masih NaN (jika ada)
df_clean = df_clean.dropna()
print('Shape setelah cleaning:', df_clean.shape)

In [ ]:
# Outlier — pertahankan 1998 dan 2020 (kejadian nyata, bukan noise)
print('Data outlier yang dipertahankan (krisis nyata):')
print(df_clean[df_clean['Year'].isin([1998, 2020])][['Year', 'GDP_Growth']])

# Simpan data cleaned
df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print('Dataset clean disimpan!')

In [ ]:
# Fitur dan target
FEATURES = ['Inflation', 'Unemployment', 'Population_Growth',
            'Exports', 'Imports', 'FDI', 'Exchange_Rate']
TARGET = 'GDP_Growth'

X = df_clean[FEATURES]
y = df_clean[TARGET]

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Train-test split (80-20), tanpa shuffle karena time series
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f'Training: {X_train.shape[0]} baris')
print(f'Testing:  {X_test.shape[0]} baris')

In [ ]:
# StandardScaler — fit hanya pada training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Simpan scaler
joblib.dump(scaler, '../models/scaler.pkl')
print('scaler.pkl disimpan!')

# Simpan split data
import pickle
with open('../models/data_split.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'features': FEATURES
    }, f)
print('data_split.pkl disimpan!')